# UdaPlay — Notebook 01: RAG Pipeline

Builds a **persistent ChromaDB vector database** from the local game dataset.
This database is the knowledge base for the AI agent in Notebook 02.

**Steps:**
1. Load 15 game JSON files via `GameJSONLoader`
2. Format documents with structured metadata
3. Initialise a persistent ChromaDB collection
4. Embed all documents with OpenAI text-embedding-ada-002
5. Demonstrate semantic search

> **Run order:** This notebook must run before `02_agent_demo.ipynb`.

In [11]:
# Make the udaplay package importable when running from notebooks/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / "src"))

In [12]:
# SQLite shim — only needed on older Udacity workspace environments
import importlib.util
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [13]:
import chromadb
from chromadb.utils import embedding_functions

from udaplay.config import Settings
from udaplay.loaders import GameJSONLoader, format_game_document, build_metadata

settings = Settings()
settings.validate()          # raises EnvironmentError if OPENAI_API_KEY missing
print("Settings loaded. OpenAI key present:", settings.has_openai_key())

Settings loaded. OpenAI key present: True


## 1 · Load Game Dataset

In [14]:
import json, pathlib

DATA_DIR = pathlib.Path().resolve().parent / "data" / "games"
loader = GameJSONLoader(DATA_DIR)

games = loader.load()
print(f"Loaded {len(games)} game records.")
print("\nSample record:")
print(json.dumps({k: v for k, v in games[0].items() if k != 'source_file'}, indent=2))

Loaded 15 game records.

Sample record:
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}


## 2 · Format Documents + Metadata

In [15]:
import os

corpus = loader.as_corpus()   # returns a Corpus of Document objects

doc_ids   = [doc.id       for doc in corpus]
documents = [doc.content  for doc in corpus]
metadatas = [doc.metadata for doc in corpus]

print(f"Prepared {len(documents)} documents.")
print("\nExample document text:")
print(documents[0])
print("\nExample metadata:")
print(json.dumps(metadatas[0], indent=2))

Prepared 15 documents.

Example document text:
Game: Gran Turismo. Platform: PlayStation 1. Genre: Racing. Publisher: Sony Computer Entertainment. Released: 1997. Description: A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

Example metadata:
{
  "title": "Gran Turismo",
  "platform": "PlayStation 1",
  "genre": "Racing",
  "publisher": "Sony Computer Entertainment",
  "release_year": 1997,
  "source_file": "001.json"
}


## 3 · Initialise Persistent ChromaDB

In [16]:
CHROMA_PATH     = pathlib.Path().resolve().parent / "chromadb"
COLLECTION_NAME = settings.collection_name

chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
print(f"ChromaDB persistent client → {CHROMA_PATH}")

ChromaDB persistent client → /Users/suleimanadebowaleojo/Claude/Projects/UdaPlay - An AI Research Agent for the Video Game Industry/chromadb


In [17]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=settings.openai_api_key,
    model_name=settings.embedding_model,
)

# Clean rebuild — delete if already exists
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection '{COLLECTION_NAME}' created.")

Deleted existing collection 'udaplay_games'.
Collection 'udaplay_games' created.


## 4 · Add Documents to ChromaDB

In [18]:
collection.add(ids=doc_ids, documents=documents, metadatas=metadatas)
print(f"Added {collection.count()} documents to '{COLLECTION_NAME}'.")

Added 15 documents to 'udaplay_games'.


## 5 · Verify & Semantic Search

In [19]:
all_docs = collection.get(include=["documents", "metadatas"])

print(f"Total documents: {len(all_docs['ids'])}\n")
print("{:<6} {:<35} {:<30} {:<6}".format("ID", "Title", "Platform", "Year"))
print("-" * 80)
for doc_id, meta in zip(all_docs["ids"], all_docs["metadatas"]):
    print("{:<6} {:<35} {:<30} {:<6}".format(
        doc_id, meta["title"][:33], meta["platform"][:28], meta["release_year"]
    ))

Total documents: 15

ID     Title                               Platform                       Year  
--------------------------------------------------------------------------------
001    Gran Turismo                        PlayStation 1                  1997  
002    Grand Theft Auto: San Andreas       PlayStation 2                  2004  
003    Gran Turismo 5                      PlayStation 3                  2010  
004    Marvel's Spider-Man                 PlayStation 4                  2018  
005    Marvel's Spider-Man 2               PlayStation 5                  2023  
006    Pokémon Gold and Silver             Game Boy Color                 1999  
007    Pokémon Ruby and Sapphire           Game Boy Advance               2002  
008    Super Mario World                   Super Nintendo Entertainment   1990  
009    Super Mario 64                      Nintendo 64                    1996  
010    Super Smash Bros. Melee             GameCube                       2001  
011    

In [20]:
def semantic_search(query: str, n: int = 3):
    r = collection.query(
        query_texts=[query], n_results=n,
        include=["documents", "metadatas", "distances"]
    )
    print(f"\nQuery: '{query}'")
    print("=" * 65)
    for rank, (doc, meta, dist) in enumerate(
        zip(r["documents"][0], r["metadatas"][0], r["distances"][0]), 1
    ):
        print(f"  Rank {rank} | Similarity {1-dist:.4f} | {meta['title']} ({meta['release_year']})")
        print(f"           | {meta['platform']} | {meta['publisher']}")

semantic_search("Nintendo racing game for the Switch")
semantic_search("Pokémon role-playing game")
semantic_search("open-world crime game Rockstar")
semantic_search("superhero action game PlayStation")
semantic_search("first 3D Mario platformer Nintendo 64")


Query: 'Nintendo racing game for the Switch'
  Rank 1 | Similarity 0.8939 | Mario Kart 8 Deluxe (2017)
           | Nintendo Switch | Nintendo
  Rank 2 | Similarity 0.8229 | Wii Sports (2006)
           | Wii | Nintendo
  Rank 3 | Similarity 0.8127 | Gran Turismo 5 (2010)
           | PlayStation 3 | Sony Computer Entertainment

Query: 'Pokémon role-playing game'
  Rank 1 | Similarity 0.8863 | Pokémon Ruby and Sapphire (2002)
           | Game Boy Advance | Nintendo
  Rank 2 | Similarity 0.8814 | Pokémon Gold and Silver (1999)
           | Game Boy Color | Nintendo
  Rank 3 | Similarity 0.7949 | Super Mario 64 (1996)
           | Nintendo 64 | Nintendo

Query: 'open-world crime game Rockstar'
  Rank 1 | Similarity 0.8451 | Grand Theft Auto: San Andreas (2004)
           | PlayStation 2 | Rockstar Games
  Rank 2 | Similarity 0.8056 | Marvel's Spider-Man (2018)
           | PlayStation 4 | Sony Interactive Entertainment
  Rank 3 | Similarity 0.8000 | Minecraft (2014)
           | Xbox O

## ✅ Summary

| Step | Result |
|------|--------|
| 15 game JSON files loaded via `GameJSONLoader` | ✅ |
| Documents formatted with 6-field metadata | ✅ |
| Persistent ChromaDB collection created | ✅ |
| OpenAI `text-embedding-ada-002` embeddings | ✅ |
| Semantic search demonstrated (5 queries) | ✅ |

The database is saved to `chromadb/` and ready for **Notebook 02**.